# Profiling ANP - Entendendo o dataset e suas propriedades

In [1]:
import polars as pl
from pathlib import Path

def carregar_particao(caminho: Path) -> pl.DataFrame:
    df = pl.read_csv(
    caminho,
    separator=";"
    )
    return df

def extrair_metadados_particao(arquivo: Path) -> dict:
    return {
        "semestre": int(arquivo.parent.name.split("=")[1]),
        "ano": int(arquivo.parent.parent.name.split("=")[1])
    }

def verificar_chave_candidata(df: pl.DataFrame) -> pl.DataFrame:
    duplicados = (
        df.select(
            "CNPJ da Revenda",
            "Produto",
            "Data da Coleta"
        )
        .filter(
            pl.col("CNPJ da Revenda").is_not_null() &
            pl.col("Produto").is_not_null() &
            pl.col("Data da Coleta").is_not_null()       
        )
        .group_by([
            "CNPJ da Revenda",
            "Produto",
            "Data da Coleta"
        ])
        .len()
        .filter(
            pl.col("len") > 1
        )
    )
    return duplicados

def analisar_particao(df: pl.DataFrame) -> dict:
    resultado = {        
        "total_linhas": df.height,
        "total_colunas": df.width,        
        "colunas": df.columns,
        "data_minima": df["Data da Coleta"].str.to_date(format="%d/%m/%Y").min(),
        "data_maxima": df["Data da Coleta"].str.to_date(format="%d/%m/%Y").max(),
        "produtos": df["Produto"].unique().sort().to_list(),
        "unidades_de_medida": df["Unidade de Medida"].unique().sort().to_list(),
        "colisoes_chave_candidata": verificar_chave_candidata(df).height,
        "duplicatas_exatas": quantidade_duplicatas_exatas(df),
        "linhas_vazias": quantidade_linhas_vazias(df),
        "nulos": quantidade_nulos(df),
    }
    return resultado

def quantidade_nulos(df: pl.DataFrame) -> dict:
    nulos = df.null_count().to_dicts()[0]

    nulos_com_valores = {}

    for col, count in nulos.items():
        if count > 0:
            nulos_com_valores[col] = count
    return nulos_com_valores

def quantidade_duplicatas_exatas(df: pl.DataFrame) -> int:
    return df.height - df.unique().height

def quantidade_linhas_vazias(df: pl.DataFrame) -> int:
    return df.filter(
        pl.all_horizontal(
            pl.all().is_null()
        )
    ).height

In [2]:
pasta_bronze = Path("../data/bronze/anp/automotivos")
arquivos = sorted(pasta_bronze.rglob("*.csv"))

print(f"Arquivos encontrados: {len(arquivos)}")

resultados = []

for arquivo in arquivos:
    metadados = extrair_metadados_particao(arquivo)
    df = carregar_particao(arquivo)
    resultado = analisar_particao(df)
    resultado_completo = metadados | resultado
    resultados.append(resultado_completo)

print(resultados)

Arquivos encontrados: 6
[{'semestre': 1, 'ano': 2023, 'total_linhas': 431576, 'total_colunas': 16, 'colunas': ['Regiao - Sigla', 'Estado - Sigla', 'Municipio', 'Revenda', 'CNPJ da Revenda', 'Nome da Rua', 'Numero Rua', 'Complemento', 'Bairro', 'Cep', 'Produto', 'Data da Coleta', 'Valor de Venda', 'Valor de Compra', 'Unidade de Medida', 'Bandeira'], 'data_minima': datetime.date(2023, 1, 2), 'data_maxima': datetime.date(2023, 6, 30), 'produtos': ['DIESEL', 'DIESEL S10', 'ETANOL', 'GASOLINA', 'GASOLINA ADITIVADA', 'GNV'], 'unidades_de_medida': ['R$ / litro', 'R$ / m³'], 'colisoes_chave_candidata': 0, 'duplicatas_exatas': 0, 'linhas_vazias': 0, 'nulos': {'Numero Rua': 107, 'Complemento': 333000, 'Bairro': 828, 'Valor de Compra': 431576}}, {'semestre': 2, 'ano': 2023, 'total_linhas': 472424, 'total_colunas': 16, 'colunas': ['Regiao - Sigla', 'Estado - Sigla', 'Municipio', 'Revenda', 'CNPJ da Revenda', 'Nome da Rua', 'Numero Rua', 'Complemento', 'Bairro', 'Cep', 'Produto', 'Data da Coleta', 